# 14a_src_test_prep_training_data

This notebook tests the `src.data_loader` pipeline by preparing the training dataset from raw BOM weather data and BITRE flight data. It does not train any models.

Output: `data/processed/ml_training_data_multiroute_hols.csv`.


In [13]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


In [14]:
from src.data_loader import (
    CITY_MAPPING,
    DEFAULT_CITIES,
    DATA_RAW,
    DATA_PROCESSED,
    download_bitre_data,
    compute_weather_features,
    prepare_training_data,
    update_all_data,
)

print('Imports complete')


Imports complete


In [15]:
# =============================================================================
# USER CONFIG
# =============================================================================
CITIES = DEFAULT_CITIES  # e.g., ['Sydney', 'Melbourne', ...]
OUTPUT_FILENAME = 'ml_training_data_multiroute_hols.csv'

# Set to True to recompute weather features from raw BOM data
FORCE_WEATHER_FEATURES = False

# Set to True to attempt downloading latest BITRE file
DOWNLOAD_BITRE = False

# Set to True to run the full update_all_data pipeline
RUN_UPDATE_ALL = False

# Optional override to use a specific BITRE Excel file
BITRE_FILE_OVERRIDE = None

# Validation controls
TEST_CITY = None  # e.g., 'Sydney' (defaults to first city)
VALIDATE_WEATHER = True
VALIDATE_BITRE = True
VALIDATE_TRAINING_DATA = True

print(f'Cities: {CITIES}')
print(f'Output: {OUTPUT_FILENAME}')


Cities: ['Sydney', 'Melbourne', 'Hobart', 'Brisbane', 'Perth', 'Adelaide']
Output: ml_training_data_multiroute_hols.csv


In [16]:
# Preflight checks
from pathlib import Path
import os

if TEST_CITY is None:
    TEST_CITY = CITIES[0]

missing_dirs = []
for city in CITIES:
    code = CITY_MAPPING[city]
    raw_dir = Path(DATA_RAW) / code
    if not raw_dir.exists():
        missing_dirs.append(str(raw_dir))

if missing_dirs:
    raise FileNotFoundError(f'Missing raw BOM weather folders: {missing_dirs}')

bitre_candidates = list(Path(DATA_RAW).glob('OTP_Time_Series_Master_Current_*.xlsx'))
if BITRE_FILE_OVERRIDE is not None:
    override_path = Path(BITRE_FILE_OVERRIDE)
    if not override_path.exists():
        raise FileNotFoundError(f'BITRE_FILE_OVERRIDE not found: {override_path}')
elif not DOWNLOAD_BITRE and not bitre_candidates:
    print('Warning: No local BITRE files found. You may need to set DOWNLOAD_BITRE=True.')

print('Preflight checks complete.')


Preflight checks complete.


In [17]:
# Validate raw BOM weather data exists
missing_dirs = []
for city in CITIES:
    code = CITY_MAPPING[city]
    raw_dir = Path(DATA_RAW) / code
    if not raw_dir.exists():
        missing_dirs.append(str(raw_dir))

if missing_dirs:
    raise FileNotFoundError(f'Missing raw BOM weather folders: {missing_dirs}')

print('Raw BOM weather folders found for all cities.')


Raw BOM weather folders found for all cities.


In [18]:
# Compute monthly weather features if missing or forced
for city in CITIES:
    code = CITY_MAPPING[city]
    features_path = Path(DATA_PROCESSED) / f'features_{code}.csv'
    if FORCE_WEATHER_FEATURES or not features_path.exists():
        print(f'Computing weather features for {city} ({code})...')
        compute_weather_features(code)
    else:
        print(f'Weather features already exist for {city} ({code}): {features_path}')


Weather features already exist for Sydney (syd): /Users/yen/code/Flight-Lateness-Australia-Prediction-System/data/processed/features_syd.csv
Weather features already exist for Melbourne (mel): /Users/yen/code/Flight-Lateness-Australia-Prediction-System/data/processed/features_mel.csv
Weather features already exist for Hobart (hba): /Users/yen/code/Flight-Lateness-Australia-Prediction-System/data/processed/features_hba.csv
Weather features already exist for Brisbane (bne): /Users/yen/code/Flight-Lateness-Australia-Prediction-System/data/processed/features_bne.csv
Weather features already exist for Perth (per): /Users/yen/code/Flight-Lateness-Australia-Prediction-System/data/processed/features_per.csv
Weather features already exist for Adelaide (adl): /Users/yen/code/Flight-Lateness-Australia-Prediction-System/data/processed/features_adl.csv


In [19]:
# Validate weather features for one city
if VALIDATE_WEATHER:
    code = CITY_MAPPING[TEST_CITY]
    features_path = Path(DATA_PROCESSED) / f'features_{code}.csv'
    if not features_path.exists():
        raise FileNotFoundError(f'Weather features not found: {features_path}')
    print(f'PASS: Weather features file exists for {TEST_CITY} ({code}).')

    df_weather = pd.read_csv(features_path)
    required_cols = [
        'year_month', 'days_in_month', 'avg_rainfall_per_day',
        'max_temperature', 'avg_max_temp', 'min_temperature', 'avg_min_temp',
        'max_daily_rainfall', 'avg_wind_speed', 'max_wind_speed',
        'avg_max_humidity', 'avg_min_humidity',
        'temp_range_mean', 'days_above_35C', 'temp_volatility',
        'rainy_days', 'heavy_rain_days', 'avg_rainfall_on_rainy_days',
        'days_high_wind', 'wind_speed_std',
        'days_high_humidity', 'extreme_weather_days',
    ]
    missing = [c for c in required_cols if c not in df_weather.columns]
    if missing:
        raise ValueError(f'Missing expected weather columns: {missing}')
    print('PASS: Weather feature columns present.')

    if df_weather.empty:
        raise ValueError('Weather feature file is empty.')
    print(f'PASS: Weather features non-empty. Rows: {len(df_weather)}')


PASS: Weather features file exists for Sydney (syd).
PASS: Weather feature columns present.
PASS: Weather features non-empty. Rows: 206


In [20]:
# Resolve BITRE file
if BITRE_FILE_OVERRIDE:
    bitre_file = BITRE_FILE_OVERRIDE
    print(f'Using BITRE file override: {bitre_file}')
elif DOWNLOAD_BITRE:
    bitre_file = download_bitre_data()
    print(f'Downloaded BITRE file: {bitre_file}')
else:
    bitre_file = None
    print('Using latest local BITRE file (if available).')


Using latest local BITRE file (if available).


In [21]:
# Validate BITRE file resolution
if VALIDATE_BITRE:
    if bitre_file is not None:
        bitre_path = Path(bitre_file)
        if not bitre_path.exists():
            raise FileNotFoundError(f'BITRE file not found: {bitre_path}')
        print('PASS: BITRE file exists.')
        if bitre_path.stat().st_size <= 0:
            raise ValueError(f'BITRE file is empty: {bitre_path}')
        print('PASS: BITRE file is non-empty.')
        print(f'BITRE validation passed: {bitre_path}')
    else:
        print('SKIP: BITRE file resolved inside prepare_training_data().')


SKIP: BITRE file resolved inside prepare_training_data().


In [22]:
# Prepare training data (no model training)
df_final = prepare_training_data(
    cities=CITIES,
    bitre_file=bitre_file,
    output_filename=OUTPUT_FILENAME,
)

print('Training data prepared.')


Loading BITRE data from: /Users/yen/code/Flight-Lateness-Australia-Prediction-System/data/raw/OTP_Time_Series_Master_Current_november_2025.xlsx
Flight records after cleaning: 16772
Merged records: 16772
Training data saved to: /Users/yen/code/Flight-Lateness-Australia-Prediction-System/data/processed/ml_training_data_multiroute_hols.csv (16772 rows)
Training data prepared.


In [23]:
# Validate training dataset output
if VALIDATE_TRAINING_DATA:
    output_path = Path(DATA_PROCESSED) / OUTPUT_FILENAME
    if not output_path.exists():
        raise FileNotFoundError(f'Training data output not found: {output_path}')
    print('PASS: Training data output file exists.')

    if df_final.empty:
        raise ValueError('Training dataset is empty.')
    print(f'PASS: Training dataset non-empty. Rows: {len(df_final)}')

    required_cols = [
        'year_month', 'airline', 'departing_port', 'arriving_port',
        'delay_rate', 'is_high_delay',
    ]
    missing = [c for c in required_cols if c not in df_final.columns]
    if missing:
        raise ValueError(f'Missing expected training columns: {missing}')
    print('PASS: Required training columns present.')

    if (df_final['delay_rate'] < 0).any() or (df_final['delay_rate'] > 1).any():
        raise ValueError('delay_rate outside [0, 1] range detected.')
    print('PASS: delay_rate within [0, 1].')

    null_counts = df_final[required_cols].isnull().sum()
    if (null_counts > 0).any():
        raise ValueError(f'Nulls in required columns: {null_counts[null_counts > 0].to_dict()}')
    print('PASS: No nulls in required columns.')


PASS: Training data output file exists.
PASS: Training dataset non-empty. Rows: 16772
PASS: Required training columns present.
PASS: delay_rate within [0, 1].
PASS: No nulls in required columns.


In [24]:
# Basic summary
output_path = Path(DATA_PROCESSED) / OUTPUT_FILENAME
print(f'Output path: {output_path}')
print(f'Rows: {len(df_final)}')
print(f"Date range: {df_final['year_month'].min()} to {df_final['year_month'].max()}")

df_final.head(10)


Output path: /Users/yen/code/Flight-Lateness-Australia-Prediction-System/data/processed/ml_training_data_multiroute_hols.csv
Rows: 16772
Date range: 2010-01 to 2025-11


,departing_port,arriving_port,airline,month,year_month,year,sectors_scheduled,sectors_flown,arrivals_on_time,arrivals_delayed,cancellations,cancellations_pct,delay_rate,is_high_delay,avg_max_humidity_dep,avg_max_temp_dep,avg_min_humidity_dep,avg_min_temp_dep,avg_rainfall_on_rainy_days_dep,avg_rainfall_per_day_dep,avg_wind_speed_dep,days_above_35C_dep,days_high_humidity_dep,days_high_wind_dep,days_in_month_dep,extreme_weather_days_dep,heavy_rain_days_dep,max_daily_rainfall_dep,max_temperature_dep,max_wind_speed_dep,min_temperature_dep,rainy_days_dep,temp_range_mean_dep,temp_volatility_dep,wind_speed_std_dep,avg_max_humidity_arr,avg_max_temp_arr,avg_min_humidity_arr,avg_min_temp_arr,avg_rainfall_on_rainy_days_arr,avg_rainfall_per_day_arr,avg_wind_speed_arr,days_above_35C_arr,days_high_humidity_arr,days_high_wind_arr,days_in_month_arr,extreme_weather_days_arr,heavy_rain_days_arr,max_daily_rainfall_arr,max_temperature_arr,max_wind_speed_arr,min_temperature_arr,rainy_days_arr,temp_range_mean_arr,temp_volatility_arr,wind_speed_std_arr,year_month_dt,month_num,n_public_holidays_total,pct_school_holiday
0,Adelaide,Brisbane,Jetstar,2010-01-01,2010-01,2010,30.0,30.0,23.0,7.0,0.0,0.000000,0.233333,0,73.419355,29.635484,26.516129,16.764516,2.75,0.354839,5.072258,5.0,1,1.0,31,6,0.0,7.0,41.0,8.57,10.7,4.0,12.870968,3.583333,1.506450,89.483871,29.664516,53.451613,21.512903,5.30,1.709677,4.334194,0.0,15,0.0,31,2,2.0,22.0,32.1,6.35,17.8,10.0,8.151613,0.873333,0.786116,2010-01-01,1,2,0.9032
1,Adelaide,Melbourne,Jetstar,2010-01-01,2010-01,2010,47.0,47.0,36.0,11.0,0.0,0.000000,0.234043,0,73.419355,29.635484,26.516129,16.764516,2.75,0.354839,5.072258,5.0,1,1.0,31,6,0.0,7.0,41.0,8.57,10.7,4.0,12.870968,3.583333,1.506450,86.451613,27.003226,32.322581,13.922581,5.36,0.864516,5.651290,6.0,7,3.0,31,12,1.0,14.0,42.5,10.14,8.6,5.0,13.080645,5.476667,1.690102,2010-01-01,1,2,0.9032
2,Adelaide,Perth,Jetstar,2010-01-01,2010-01,2010,29.0,29.0,26.0,3.0,0.0,0.000000,0.103448,0,73.419355,29.635484,26.516129,16.764516,2.75,0.354839,5.072258,5.0,1,1.0,31,6,0.0,7.0,41.0,8.57,10.7,4.0,12.870968,3.583333,1.506450,70.354839,35.048387,20.806452,18.835484,0.00,0.000000,5.922581,17.0,2,0.0,31,17,0.0,0.0,43.2,7.67,12.4,0.0,16.212903,3.456667,0.934048,2010-01-01,1,2,0.9032
3,Adelaide,Sydney,Jetstar,2010-01-01,2010-01,2010,59.0,59.0,40.0,19.0,0.0,0.000000,0.322034,1,73.419355,29.635484,26.516129,16.764516,2.75,0.354839,5.072258,5.0,1,1.0,31,6,0.0,7.0,41.0,8.57,10.7,4.0,12.870968,3.583333,1.506450,89.709677,27.958065,52.419355,20.587097,3.30,0.851613,6.013226,1.0,18,2.0,31,7,1.0,11.6,42.5,9.50,15.6,8.0,7.370968,3.963333,1.468147,2010-01-01,1,2,0.9032
4,Brisbane,Adelaide,Jetstar,2010-01-01,2010-01,2010,29.0,29.0,23.0,6.0,0.0,0.000000,0.206897,0,89.483871,29.664516,53.451613,21.512903,5.30,1.709677,4.334194,0.0,15,0.0,31,2,2.0,22.0,32.1,6.35,17.8,10.0,8.151613,0.873333,0.786116,73.419355,29.635484,26.516129,16.764516,2.75,0.354839,5.072258,5.0,1,1.0,31,6,0.0,7.0,41.0,8.57,10.7,4.0,12.870968,3.583333,1.506450,2010-01-01,1,2,0.9032
5,Brisbane,Sydney,Jetstar,2010-01-01,2010-01,2010,89.0,83.0,74.0,9.0,6.0,6.741573,0.108434,0,89.483871,29.664516,53.451613,21.512903,5.30,1.709677,4.334194,0.0,15,0.0,31,2,2.0,22.0,32.1,6.35,17.8,10.0,8.151613,0.873333,0.786116,89.709677,27.958065,52.419355,20.587097,3.30,0.851613,6.013226,1.0,18,2.0,31,7,1.0,11.6,42.5,9.50,15.6,8.0,7.370968,3.963333,1.468147,2010-01-01,1,2,0.9032
6,Hobart,Melbourne,Jetstar,2010-01-01,2010-01,2010,131.0,131.0,104.0,27.0,0.0,0.000000,0.206107,0,80.870968,23.803226,34.612903,12.264516,1.52,0.245161,5.320968,2.0,0,0.0,31,2,0.0,5.2,36.0,7.53,7.2,5.0,11.538710,4.280000,0.985090,86.451613,27.003226,32.322581,13.922581,5.36,0.864516,5.651290,6.0,7,3.0,31,12,1.0,14.0,42.5,10.14,8.6,5.0,13.080645,5.476667,1.690102,2010-01-01,1,2,0.9032
7,Hobart,Sydney,Jetstar,2010-01-01,2010-01,2010,59.0,59.0,51.0,8.0,0.0,0.000000,0.135593,0,80.870968,23.803226,34.612903,12.264516,1.52,0.245161,5.320968,2.0,0,0.0,31,2,0.0,5.2,36.0,7.53,7.2,5.0,